In [2]:
from abc import ABC, abstractmethod
from typing import Optional, List, Type, Tuple, Dict
import math

import numpy as np
from matplotlib import pyplot as plt
from matplotlib.axes._axes import Axes
import torch
import torch.nn as nn
import torch.distributions as D
from torch.func import vmap, jacrev
from tqdm import tqdm
import seaborn as sns
from sklearn.datasets import make_moons, make_circles
from torchvision import datasets, transforms
from torchvision.utils import make_grid

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cpu


## Loader

In [4]:
import pandas as pd

def load_npz_to_dataframe(filename: str, include_targets: bool = True):
    ''' Reconstructs a DataFrame from a saved npz file '''
    data = np.load(filename)
    df = pd.DataFrame(data['features'], columns=data['feature_names'])
    df['targets'] = [t for t in data['targets']]
    
    return df

poseTraject_df = load_npz_to_dataframe("pose_traject_dataset_2000x50_0.npz")
#print(poseTraject_df.head())

In [5]:
# 1. One-hot encode : This creates the 3D one-hot vector part = {Accel, Const, Decel} [2]
part_onehot = pd.get_dummies(poseTraject_df['part_enum'].astype(int), prefix='part').values

# 2. Assemble the final Context Vector C (9 scalars total) [2]
# C = [s_goal_x, s_goal_y, v_const, accel, part_0, part_1, part_2, q_init_x, q_init_y]
C = np.concatenate([
    poseTraject_df[['s_goal_x', 's_goal_y', 'v_const', 'accel']].values,
    part_onehot,
    poseTraject_df[['q_init_x', 'q_init_y']].values
], axis=1)

# 3. Process the Action Vector A (2D trajectories) [1]
# The 'targets' column contains arrays of shape (50, 7) [5]
# We stack them and keep only (x, y) which are indices 0 and 1 [6]
targets_raw = np.stack(poseTraject_df['targets'].values) 
A_2d = targets_raw[:, :, 0:2] # Shape: (batch_size, 50 steps, 2 variables)

# Transpose A for 1D Convolution [batch, channels, length] [3]
# Final shape should be (batch_size, 2, 50)
A = np.transpose(A_2d, (0, 2, 1))

# 4. Prepare PyTorch Tensors for Flow Matching [7]
context_tensor = torch.tensor(C, dtype=torch.float32)
action_tensor = torch.tensor(A, dtype=torch.float32)

# context_tensor: Shape (N, 9) 
# Columns: [s_goal_x, s_goal_y, v_const, accel, part_accel, part_const, part_decel, q_init_x, q_init_y]
# Purpose: Provides physical constraints and the current observed state to guide the flow.

# action_tensor: Shape (N, 2, 50) 
# Channels: [x_pose_sequence, y_pose_sequence]
# Purpose: The target 2D trajectory (50 steps) for the model to learn via Flow Matching.

print(f"Context Vector C (Guided Context) shape: {context_tensor.shape}") # Expected: (N, 9)
print(f"Action Vector A (Trajectory) shape: {action_tensor.shape}")     # Expected: (N, 2, 50)

# print first 5 contexts and actions
print("\nFirst 5 Context Vectors C:")
print(context_tensor[:5])
print("First 5 Action Vectors A:")
print(action_tensor[:5])

Context Vector C (Guided Context) shape: torch.Size([100000, 9])
Action Vector A (Trajectory) shape: torch.Size([100000, 2, 50])

First 5 Context Vectors C:
tensor([[1.2148, 3.5410, 0.1300, 0.0344, 0.0000, 1.0000, 0.0000, 1.0635, 3.0996],
        [1.2148, 3.5410, 0.1300, 0.0344, 0.0000, 1.0000, 0.0000, 0.1769, 0.5156],
        [1.2148, 3.5410, 0.1300, 0.0344, 0.0000, 1.0000, 0.0000, 0.3252, 0.9482],
        [1.2148, 3.5410, 0.1300, 0.0344, 0.0000, 0.0000, 1.0000, 1.1660, 3.3984],
        [1.2148, 3.5410, 0.1300, 0.0344, 0.0000, 1.0000, 0.0000, 0.0867, 0.2527]])
First 5 Action Vectors A:
tensor([[[1.0635, 1.0645, 1.0664, 1.0684, 1.0693, 1.0713, 1.0732, 1.0752,
          1.0762, 1.0781, 1.0801, 1.0820, 1.0830, 1.0850, 1.0869, 1.0879,
          1.0898, 1.0918, 1.0938, 1.0947, 1.0967, 1.0986, 1.1006, 1.1016,
          1.1035, 1.1055, 1.1064, 1.1084, 1.1104, 1.1123, 1.1133, 1.1152,
          1.1172, 1.1191, 1.1201, 1.1221, 1.1240, 1.1250, 1.1270, 1.1289,
          1.1309, 1.1318, 1.1338, 1.

## TrajectorySampler

In [ ]:
import torch
import numpy as np
from torch.utils.data import Dataset

class TrajectorySampler(Dataset):
    def __init__(self, file_path):
        data = np.load(file_path)
        self.contexts = torch.from_numpy(data['context']).float()  # Expected (N, 9)
        self.actions = torch.from_numpy(data['actions']).float()    # Expected (N, 2, 50)
        
        # Calculate normalization statistics for the actions
        self.min_val = self.actions.view(-1, 2).min(dim=0)[0].view(1, 2, 1)
        self.max_val = self.actions.view(-1, 2).max(dim=0)[0].view(1, 2, 1)


    def __len__(self):
        return len(self.actions)


    def normalize(self, x):
        return 2 * (x - self.min_val) / (self.max_val - self.min_val + 1e-8) - 1


    def __getitem__(self, idx):
        context = self.contexts[idx]
        trajectory = self.actions[idx]
        
        # Rigorous shape validation
        assert context.shape == (9,), f"Expected context (9,), got {context.shape}"
        assert trajectory.shape == (2, 50), f"Expected action (2, 50), got {trajectory.shape}"
        
        return context, self.normalize(trajectory)


In [ ]:
import torch.nn as nn


class TrajectoryUNet1D(nn.Module):
    def __init__(self, context_dim=9, hidden_dim=256):
        super().__init__()
        # Time and Context Embeddings
        self.time_mlp = nn.Sequential(nn.Linear(1, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, hidden_dim))
        self.context_mlp = nn.Sequential(nn.Linear(context_dim, hidden_dim), nn.SiLU(), nn.Linear(hidden_dim, hidden_dim))
        
        # Symmetric U-Net Structure
        self.enc = nn.Sequential(nn.Conv1d(2, hidden_dim, 3, padding=1), nn.BatchNorm1d(hidden_dim), nn.SiLU())
        self.mid = nn.Sequential(nn.Conv1d(hidden_dim, hidden_dim, 3, padding=1), nn.SiLU())
        self.dec = nn.Sequential(nn.Conv1d(hidden_dim, hidden_dim, 3, padding=1), nn.BatchNorm1d(hidden_dim), nn.SiLU())
        self.final_conv = nn.Conv1d(hidden_dim, 2, 1)


    def forward(self, x, context, t):
        """
        x: (batch, 2, 50) | context: (batch, 9) | t: (batch, 1)
        """
        # Generate embeddings
        t_emb = self.time_mlp(t).unsqueeze(-1)         # (B, H, 1)
        c_emb = self.context_mlp(context).unsqueeze(-1) # (B, H, 1)
        emb = t_emb + c_emb # Combined guidance
        
        # Forward pass with broadcasting integration
        h = self.enc(x) + emb.repeat(1, 1, x.shape[-1]) # Explicit expansion across temporal dimension
        h = self.mid(h) + emb
        h = self.dec(h) + emb
        return self.final_conv(h)
